# 🧪 Pre-Training Architecture Validation

Before we commit to the long 5+ hour training process, this notebook proves that:
1. The raw `SpeechT5` autoencoder works beautifully when strictly using its native latent space.
2. The `Wav2Vec2` encoder works perfectly and captures the English content accurately (proven via ASR).
3. **Bridging them currently sounds like noise**, explicitly proving exactly why the fine-tuning script is mandatory.


In [2]:
import sys, os
sys.path.append(os.path.abspath('../../'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML
from datasets import load_dataset
import dataset_loader
from encoders import Wav2VecSpeechT5Encoder
from transformers import SpeechT5Processor, SpeechT5ForSpeechToSpeech, SpeechT5HifiGan, Wav2Vec2ForCTC, Wav2Vec2Processor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


Device: cuda


## 1. Extract a Real Speaker Embedding & Load English Audio


In [3]:
print('Loading a real speaker embedding from CMU Arctic...')
embeddings_dataset = load_dataset('Matthijs/cmu-arctic-xvectors', split='validation')
real_speaker_embedding = torch.tensor(embeddings_dataset[7306]['xvector']).unsqueeze(0).to(DEVICE)

print('\nLoading raw datasets (fleurs, 1 sample)...')
datasets = dataset_loader.load_data(
    lang=['en'],
    split='train',
    dataset=['fleurs'],
    num_samples=1
)
en_ds = datasets['en']
item = en_ds[0]['audio']
en_audio = np.array(item['array'], dtype=np.float32)
en_sr = item['sampling_rate']

print(f"\nRaw EN shape: {en_audio.shape}, SR: {en_sr}")
display(HTML("<b>Original Raw English Audio:</b>"))
display(Audio(en_audio, rate=en_sr))


Loading a real speaker embedding from CMU Arctic...


num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.



Loading raw datasets (fleurs, 1 sample)...
Loading google/fleurs (en_us) from local storage: /media/zawiatgf/New Volume/Personal Files/Abdurrahman Zawia/University/Grad Project/Speech-To-Speech-Model/datasets/fleurs/en_us/train...
Slicing loaded dataset (0:1)...
Validating en (checking audio & uniqueness)...
  1 valid unique IDs found.
Final Count: 1 common valid samples.

Raw EN shape: (108800,), SR: 16000


## 2. Test Native SpeechT5 (To prove its components are healthy)
Here we use the *pure* SpeechT5 pipeline with its own internal convolutional feature extractor.


In [4]:
print('Loading pure SpeechT5 model...')
pure_processor = SpeechT5Processor.from_pretrained('microsoft/speecht5_vc')
pure_model = SpeechT5ForSpeechToSpeech.from_pretrained('microsoft/speecht5_vc').to(DEVICE)
vocoder = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan').to(DEVICE)

inputs = pure_processor(audio=en_audio, sampling_rate=en_sr, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    pure_speech = pure_model.generate_speech(inputs['input_values'], real_speaker_embedding, vocoder=vocoder)

display(HTML("<b>Native SpeechT5 Auto-Encoded Audio (Sounds clear):</b>"))
display(Audio(pure_speech.cpu().numpy(), rate=16000))


Loading pure SpeechT5 model...


## 3. The Latent Mismatch: Decoding Wav2Vec2 via SpeechT5
Now we pass the exact same audio through `Wav2Vec2`. It creates an entirely different type of hidden state vector. When the pre-trained `SpeechT5` attempts to decode it, it fails fundamentally—which is why the audio is fuzzy!


In [5]:
print('Initializing Wav2VecSpeechT5Encoder...')
w2v_encoder = Wav2VecSpeechT5Encoder(load_decoder=True)
w2v_encoder.speecht5_model.eval()

src_hidden = w2v_encoder.encode(en_audio, sr=en_sr)
print(f"Wav2Vec2 encoded hidden states: {src_hidden.shape}")

w2v_speech = w2v_encoder.decode(
    hidden_states=src_hidden,
    speaker_embedding=real_speaker_embedding,
    threshold=0.3,
    minlenratio=0.5,
    maxlenratio=6.0
)
display(HTML("<b>Wav2Vec2 States blindly decoded by SpeechT5 (Sounds completely fuzzy without fine-tuning!):</b>"))
display(Audio(w2v_speech, rate=16000))


Initializing Wav2VecSpeechT5Encoder...
[Wav2VecSpeechT5Encoder] Loading Wav2Vec2 processor from 'facebook/wav2vec2-base-960h'...
[Wav2VecSpeechT5Encoder] Loading Wav2Vec2 model from 'facebook/wav2vec2-base-960h'...


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[Wav2VecSpeechT5Encoder] Loading SpeechT5 model from 'microsoft/speecht5_vc'...
[Wav2VecSpeechT5Encoder] Loading HiFi-GAN vocoder from 'microsoft/speecht5_hifigan'...
Wav2Vec2 encoded hidden states: torch.Size([1, 339, 768])


## 4. PROOF that Wav2Vec2 worked flawlessly (ASR Test)
To legally prove that the Wav2Vec2 encoder did not break the audio, we pass those *exact same `src_hidden` fuzzy states* into ASR (Speech-to-Text). If it prints out the correct English text, then the encoder represents the audio perfectly, entirely justifying your Fine-Tuning script to align the gap!


In [6]:
print('Loading Wav2Vec2 CTC Head...')
asr_head = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h').lm_head.to(DEVICE)
w2v_processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')

# The CTC head takes the hidden states and maps them to vocabulary logits
with torch.no_grad():
    logits = asr_head(src_hidden.to(DEVICE))
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = w2v_processor.batch_decode(predicted_ids)[0]

print(f"\n📝 Speech-To-Text Transcription of the Wav2Vec2 Hidden States:\n\"{transcription}\"")


Loading Wav2Vec2 CTC Head...


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📝 Speech-To-Text Transcription of the Wav2Vec2 Hidden States:
"A TORNADO IS A SPINNING COLUMN OF VERY LOW PRESSURE AIR WHICH SUCKS THE SURROUNDING AIR INWARD AND UPWARD"


## 5. Test Preprocessed Dataset (Target Reconstruction)
Finally, we load a sample from the **preprocessed target dataset** (the German side). These are stored as 80-bin log-mel spectrograms. We pass one through the Vocoder to ensure that the target data is correctly formatted and reconstructible.

In [7]:
from datasets import load_from_disk

# Path to the preprocessed German (target) dataset
target_ds_path = '../../datasets/processed_speecht5_wav2vec_en_de_v3/de/'
target_dataset = load_from_disk(target_ds_path)

print(f"Loaded target dataset with {len(target_dataset)} samples.")

# Take a sample (e.g., the first one)
sample_id = 0
sample_target_mel = torch.tensor(target_dataset[sample_id]['audio']).to(DEVICE)
print(f"Target Mel Spectrogram shape: {sample_target_mel.shape}")

# Decoding / Vocoding: Reconstruct audio from the mel spectrogram
with torch.no_grad():
    # Vocoder expects (Batch, Time, 80)
    reconstructed_target_audio = w2v_encoder.vocoder(sample_target_mel.unsqueeze(0)).squeeze().cpu().numpy()

display(HTML(f"<b>Reconstructed Target Audio (Sample {sample_id}) from Preprocessed Dataset:</b>"))
display(Audio(reconstructed_target_audio, rate=16000))


Loaded target dataset with 13438 samples.
Target Mel Spectrogram shape: torch.Size([259, 80])
